# SME Loan Onboarding Agent — End-to-End LangGraph Notebook V1

This notebook implements a local, GitHub-ready prototype of the architecture:

**Application → LangGraph → Specialist Agents → RAG/Vector DB + Tools → Financial/Risk Analysis → Policy Engine → HITL → Credit Memo / Decision**

### Design principles
- LangGraph orchestrates the workflow and maintains state.
- RAG supplies grounded lending-policy/SOP context.
- ChromaDB acts as the local vector database.
- Claude API is optional; without a key, deterministic fallback reasoning keeps the notebook runnable.
- Financial calculations, risk rules and policy gates are deterministic Python, not LLM-generated numbers.
- HITL is implemented with LangGraph `interrupt()` and `Command`.
- External banking/KYC/bureau/GST systems are represented by typed mock tools so they can later be replaced with real APIs/MCP servers.

## 1. Install dependencies

In [ ]:
%pip install -qU langgraph langchain-core langchain-anthropic anthropic chromadb pandas numpy pydantic python-dotenv scikit-learn

## 2. Imports and configuration

In [ ]:
import os, json, uuid
from typing import Any, Dict, List, Optional, TypedDict
import pandas as pd
from dotenv import load_dotenv
load_dotenv()
APP_ID=str(uuid.uuid4())
USE_CLAUDE=bool(os.getenv("ANTHROPIC_API_KEY"))
CLAUDE_MODEL=os.getenv("CLAUDE_MODEL", "claude-sonnet-4-5")
print("Run ID:", APP_ID, "| Claude:", USE_CLAUDE)
if USE_CLAUDE:
    from langchain_anthropic import ChatAnthropic
    llm=ChatAnthropic(model=CLAUDE_MODEL, temperature=0, max_tokens=1200)
else:
    llm=None

## 3. Synthetic SME application

In [ ]:
application={
 "application_id":"SME-10001","applicant_name":"ABC Distributors Pvt Ltd","entity_type":"Private Limited",
 "industry":"FMCG Distribution","business_vintage_years":8,"requested_amount":2500000,"tenure_months":36,
 "purpose":"Working Capital","annual_turnover":48000000,"annual_ebitda":4200000,"existing_debt":12000000,
 "promoter_obligations":2400000,"current_assets":14500000,"current_liabilities":8500000,
 "annual_interest":1200000,"annual_principal":2400000,"avg_monthly_business_inflows":3820000,
 "avg_monthly_business_outflows":3140000,"cashflow_volatility":0.12,"bureau_score":748,
 "business_dpd":0,"promoter_dpd":0,"recent_bureau_enquiries":2,"gst_turnover":47500000,
 "gst_filings_current":True,"fraud_flags":[]
}
pd.Series(application)

## 4. Local RAG knowledge base

In [ ]:
knowledge_docs=[
 {"id":"policy_001","title":"SME Credit Policy","text":"SME applicants should have a minimum business vintage of 3 years. Applications require KYC/KYB completion, satisfactory bureau conduct, financial assessment and evidence of repayment capacity. Policy exceptions must be referred to an authorized credit officer."},
 {"id":"policy_002","title":"KYC and KYB SOP","text":"Verify the identity of relevant individuals and the legal existence of the business. Validate PAN, GSTIN/CIN where applicable, ownership information and required screening. Conflicting identity or entity information requires review before credit commitment."},
 {"id":"policy_003","title":"Credit Assessment SOP","text":"Credit assessment should consider turnover, profitability, leverage, liquidity, cash-flow stability, existing debt service and bureau conduct. DSCR and other ratios must be calculated from source data rather than generated by an LLM."},
 {"id":"policy_004","title":"Human Approval Policy","text":"Cases with policy exceptions, material data conflicts, elevated fraud indicators or other configured risk triggers must be reviewed by a credit officer before approval."},
 {"id":"template_001","title":"Credit Memo Template","text":"A credit memo should contain applicant profile, requested facility, financial summary, bureau findings, cash-flow observations, key strengths, key concerns, policy outcome, conditions and recommendation."}
]
len(knowledge_docs)

## 5. Build Chroma vector DB

In [ ]:
import chromadb
from langchain_core.tools import tool
client=chromadb.PersistentClient(path="./chroma_sme_loan")
collection=client.get_or_create_collection(name="sme_credit_knowledge")
old=collection.get()
if old.get("ids"): collection.delete(ids=old["ids"])
collection.add(ids=[d["id"] for d in knowledge_docs],documents=[d["text"] for d in knowledge_docs],metadatas=[{"title":d["title"]} for d in knowledge_docs])
print("Vector DB documents:", collection.count())

@tool
def retrieve_credit_policy(query:str, top_k:int=3)->str:
    """Retrieve relevant SME credit-policy and SOP context."""
    r=collection.query(query_texts=[query],n_results=top_k)
    docs=r.get("documents",[[]])[0]; metas=r.get("metadatas",[[]])[0]
    return "\n\n".join(f"[{m.get('title','Knowledge')}] {d}" for d,m in zip(docs,metas)) or "No relevant policy context found."

print(retrieve_credit_policy.invoke({"query":"SME loan approval requirements, KYC, DSCR and human approval"}))

## 6. Banking domain tools

In [ ]:
@tool
def verify_kyb(application_id:str)->dict:
    """Mock KYB verification."""
    return {"kyb_status":"PASS","pan_verified":True,"gstin_verified":True,"entity_verified":True,"ownership_check":"PASS","screening":"CLEAR"}

@tool
def get_credit_bureau(application_id:str)->dict:
    """Mock business/promoter credit bureau result."""
    return {"bureau_score":application["bureau_score"],"business_dpd":application["business_dpd"],"promoter_dpd":application["promoter_dpd"],"recent_enquiries":application["recent_bureau_enquiries"],"outstanding_exposure":application["existing_debt"]}

@tool
def get_gst_data(application_id:str)->dict:
    """Mock GST summary."""
    return {"gst_turnover":application["gst_turnover"],"filings_current":application["gst_filings_current"],"turnover_variance_pct":round((application["gst_turnover"]-application["annual_turnover"])/application["annual_turnover"],4)}

@tool
def get_bank_cashflow(application_id:str)->dict:
    """Mock consented banking/cash-flow summary."""
    surplus=application["avg_monthly_business_inflows"]-application["avg_monthly_business_outflows"]
    return {"avg_monthly_inflows":application["avg_monthly_business_inflows"],"avg_monthly_outflows":application["avg_monthly_business_outflows"],"cashflow_volatility":application["cashflow_volatility"],"monthly_surplus":surplus}

@tool
def calculate_financial_metrics(application_id:str)->dict:
    """Deterministic financial metrics; demo thresholds only."""
    current_ratio=application["current_assets"]/application["current_liabilities"]
    debt_to_ebitda=application["existing_debt"]/application["annual_ebitda"]
    proposed_debt_service=application["requested_amount"]/(application["tenure_months"]/12)
    dscr=application["annual_ebitda"]/(application["annual_interest"]+application["annual_principal"]+proposed_debt_service)
    return {"current_ratio":round(current_ratio,2),"debt_to_ebitda":round(debt_to_ebitda,2),"annual_ebitda":application["annual_ebitda"],"dscr_demo":round(dscr,2)}

@tool
def run_fraud_checks(application_id:str)->dict:
    """Mock deterministic fraud/anomaly checks."""
    return {"fraud_risk":"LOW" if not application["fraud_flags"] else "HIGH","flags":application["fraud_flags"]}

## 7. Deterministic policy engine

In [ ]:
def run_policy_engine(app,kyb,bureau,gst,financials,fraud):
    checks=[
      {"rule":"Business vintage >= 3 years","passed":app["business_vintage_years"]>=3,"value":app["business_vintage_years"]},
      {"rule":"Bureau score >= 700","passed":bureau["bureau_score"]>=700,"value":bureau["bureau_score"]},
      {"rule":"Business DPD = 0","passed":bureau["business_dpd"]==0,"value":bureau["business_dpd"]},
      {"rule":"GST filings current","passed":gst["filings_current"],"value":gst["filings_current"]},
      {"rule":"KYB PASS","passed":kyb["kyb_status"]=="PASS","value":kyb["kyb_status"]},
      {"rule":"Fraud risk not HIGH","passed":fraud["fraud_risk"]!="HIGH","value":fraud["fraud_risk"]},
      {"rule":"DSCR demo >= 1.20","passed":financials["dscr_demo"]>=1.20,"value":financials["dscr_demo"]}
    ]
    failed=[c for c in checks if not c["passed"]]
    outcome="REFER" if failed else "PASS"
    return {"outcome":outcome,"checks":checks,"failed_rules":failed}

## 8. LangGraph state

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from typing import TypedDict

class LoanState(TypedDict, total=False):
    application: Dict[str,Any]; request:str; kyb:Dict[str,Any]; bureau:Dict[str,Any]; gst:Dict[str,Any]
    cashflow:Dict[str,Any]; financials:Dict[str,Any]; fraud:Dict[str,Any]; rag_context:str
    agent_findings:Dict[str,Any]; policy_result:Dict[str,Any]; credit_memo:str; hitl_decision:Dict[str,Any]
    final_status:str; audit:List[Dict[str,Any]]

def audit_event(state,event,details=None):
    a=list(state.get("audit",[])); a.append({"event":event,"details":details or {}}); return a

def llm_text(system,user):
    if llm is None: return ""
    return llm.invoke([("system",system),("human",user)]).content

## 9. LangGraph specialist nodes

In [ ]:
def intake_node(state):
    app=state["application"]
    return {"agent_findings":{"intake":{"applicant":app["applicant_name"],"requested_amount":app["requested_amount"],"purpose":app["purpose"]}},"audit":audit_event(state,"intake_completed")}

def kyb_node(state):
    r=verify_kyb.invoke({"application_id":state["application"]["application_id"]}); return {"kyb":r,"audit":audit_event(state,"kyb_completed",r)}

def bureau_node(state):
    r=get_credit_bureau.invoke({"application_id":state["application"]["application_id"]}); return {"bureau":r,"audit":audit_event(state,"bureau_completed",r)}

def gst_node(state):
    r=get_gst_data.invoke({"application_id":state["application"]["application_id"]}); return {"gst":r,"audit":audit_event(state,"gst_completed",r)}

def cashflow_node(state):
    r=get_bank_cashflow.invoke({"application_id":state["application"]["application_id"]}); return {"cashflow":r,"audit":audit_event(state,"cashflow_completed",r)}

def financial_node(state):
    r=calculate_financial_metrics.invoke({"application_id":state["application"]["application_id"]}); return {"financials":r,"audit":audit_event(state,"financial_analysis_completed",r)}

def fraud_node(state):
    r=run_fraud_checks.invoke({"application_id":state["application"]["application_id"]}); return {"fraud":r,"audit":audit_event(state,"fraud_completed",r)}

def rag_node(state):
    c=retrieve_credit_policy.invoke({"query":"SME loan approval requirements KYC DSCR exceptions human approval","top_k":4}); return {"rag_context":c,"audit":audit_event(state,"rag_retrieval_completed")}

def policy_node(state):
    r=run_policy_engine(state["application"],state["kyb"],state["bureau"],state["gst"],state["financials"],state["fraud"])
    return {"policy_result":r,"audit":audit_event(state,"policy_completed",{"outcome":r["outcome"],"failed_rules":r["failed_rules"]})}

def credit_memo_node(state):
    app=state["application"]; p=state["policy_result"]
    evidence={"application":app,"kyb":state["kyb"],"bureau":state["bureau"],"gst":state["gst"],"cashflow":state["cashflow"],"financials":state["financials"],"fraud":state["fraud"],"policy":p}
    if llm is not None:
        memo=llm_text("You are a banking credit-memo analyst. Use only supplied evidence.",f"Create a concise credit memo with profile, facility, financials, bureau, cash flow, strengths, concerns, policy outcome and next step. RAG context:\n{state['rag_context']}\nEvidence:\n{json.dumps(evidence,indent=2)}")
    else:
        memo=f"""SME CREDIT MEMO — {app['application_id']}\nApplicant: {app['applicant_name']}\nRequested facility: ₹{app['requested_amount']:,.0f}\nPurpose: {app['purpose']}\nTurnover: ₹{app['annual_turnover']:,.0f}\nEBITDA: ₹{app['annual_ebitda']:,.0f}\nBureau score: {state['bureau']['bureau_score']}\nBusiness DPD: {state['bureau']['business_dpd']}\nMonthly cash surplus: ₹{state['cashflow']['monthly_surplus']:,.0f}\nCurrent ratio: {state['financials']['current_ratio']}\nDebt/EBITDA: {state['financials']['debt_to_ebitda']}\nDemo DSCR: {state['financials']['dscr_demo']}\nPolicy outcome: {p['outcome']}\nFailed rules: {p['failed_rules']}\nNext step: apply the configured credit-officer/HITL workflow."""
    return {"credit_memo":memo.strip(),"audit":audit_event(state,"credit_memo_generated",{"llm_used":llm is not None})}

def human_review_node(state):
    d=interrupt({"type":"credit_officer_review","application_id":state["application"]["application_id"],"policy_outcome":state["policy_result"]["outcome"],"failed_rules":state["policy_result"]["failed_rules"],"credit_memo":state["credit_memo"],"allowed_actions":["approve","request_info","reject"]})
    return {"hitl_decision":d,"audit":audit_event(state,"human_review_completed",d)}

def final_decision_node(state):
    action=state.get("hitl_decision",{}).get("action","request_info")
    status={"approve":"APPROVED","reject":"REJECTED","request_info":"MORE_INFORMATION_REQUIRED"}.get(action,"MORE_INFORMATION_REQUIRED")
    return {"final_status":status,"audit":audit_event(state,"final_decision_recorded",{"status":status})}

## 10. Build and compile the graph

In [ ]:
builder=StateGraph(LoanState)
for name,node in [("intake",intake_node),("kyb",kyb_node),("bureau",bureau_node),("gst",gst_node),("cashflow",cashflow_node),("financial",financial_node),("fraud",fraud_node),("rag",rag_node),("policy",policy_node),("credit_memo",credit_memo_node),("human_review",human_review_node),("final_decision",final_decision_node)]:
    builder.add_node(name,node)
builder.add_edge(START,"intake")
for n in ["kyb","bureau","gst","cashflow","financial","fraud","rag"]: builder.add_edge("intake",n); builder.add_edge(n,"policy")
builder.add_edge("policy","credit_memo"); builder.add_edge("credit_memo","human_review"); builder.add_edge("human_review","final_decision"); builder.add_edge("final_decision",END)
memory=MemorySaver()
graph=builder.compile(checkpointer=memory)
print("LangGraph compiled.")

## 11. Execute — graph pauses at HITL

In [ ]:
initial_state={"application":application,"request":"Assess this SME working-capital loan application.","audit":[]}
config={"configurable":{"thread_id":application["application_id"]}}
paused=graph.invoke(initial_state,config)
print("HITL interrupt:")
print(json.dumps(paused.get("__interrupt__",[]),indent=2,default=str)[:6000])

## 12. Review the generated credit memo

In [ ]:
state=graph.get_state(config).values
print(state["credit_memo"])

## 13. Resume after human approval

In [ ]:
decision={"action":"approve","comment":"Credit officer reviewed the evidence.","reviewer":"demo_credit_officer"}
final_result=graph.invoke(Command(resume=decision),config)
print("Final status:", final_result.get("final_status"))

## 14. Audit trail and final summary

In [ ]:
final_state=graph.get_state(config).values
for i,e in enumerate(final_state.get("audit",[]),1): print(f"{i}. {e['event']}: {e['details']}")
summary={"application_id":application["application_id"],"policy_outcome":final_state["policy_result"]["outcome"],"failed_rules":final_state["policy_result"]["failed_rules"],"human_decision":final_state.get("hitl_decision"),"final_status":final_state.get("final_status"),"llm_used":llm is not None,"rag_used":True,"vector_db":"ChromaDB","orchestrator":"LangGraph"}
pd.Series(summary)

## 15. Production migration path

This notebook is deliberately executable without enterprise infrastructure.

**Replace in the next version:**
- `MemorySaver` → PostgreSQL LangGraph checkpointer
- Mock KYC/bureau/GST/banking tools → real APIs or MCP servers
- Local Chroma → production vector store if required
- Notebook → FastAPI service + Streamlit HITL UI
- Demo policy thresholds → institution-approved credit policy
- Fallback memo → structured LLM output with validation
- Simple audit list → immutable audit tables

**Important:** the financial thresholds and DSCR calculation are demonstration logic only and must not be used for real lending decisions.